# 05 - Track and register the saved model

Record the completed study, version raw data with DVC, and register the saved bundle in local MLflow. No training or Azure upload happens here. Run `bash scripts/setup_local.sh` first, then restart the kernel.

In [1]:
from pathlib import Path
import json
import platform
import sys

assert platform.system() == "Linux", "Use the WSL kernel."
assert Path(sys.prefix).name == "mlop-pj", "Select the project environment."
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if p.name == "MLOP-PJ" and (p / "README.md").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from importlib import metadata
from tracking.record import record_model
for package in ["mlflow", "dvc", "fastapi", "httpx"]:
    print(package, metadata.version(package))


/home/dew_perawis/.venvs/mlop-pj/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


mlflow 3.17.0
dvc 3.67.1
fastapi 0.143.0
httpx 0.28.1


/home/dew_perawis/.venvs/mlop-pj/lib/python3.13/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


## 1. Record the saved work

Inputs: the bundle, golden profiles, reports, and dependency lock. DVC creates a local cache and `data/raw.dvc`; initialization may stage its own metadata. MLflow creates a local database and artifact store. No Git commit or push is made. This records an earlier offline study, not a new training run.

In [2]:
registry = record_model(ROOT)
print(json.dumps(registry, indent=2))


Initialized DVC repository.

You can now commit the changes to git.

+---------------------------------------------------------------------+
|                                                                     |
|        DVC has enabled anonymous aggregate usage analytics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|             <https://dvc.org/doc/user-guide/analytics>              |
|                                                                     |
+---------------------------------------------------------------------+

What's next?
------------
- Check out the documentation: <https://dvc.org/doc>
- Get help and share ideas: <https://dvc.org/chat>
- Star us on GitHub: <https://github.com/treeverse/dvc>


⠋ Checking graph



To track the changes with git, run:

	git add data/raw.dvc

To enable auto staging, run:

	dvc config core.autostage true


2026/10/11 18:46:15 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/11 18:46:15 INFO mlflow.store.db.utils: Updating database tables
/home/dew_perawis/.venvs/mlop-pj/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/11 18:46:35 WARNING mlflow.pyfunc: Passing a Python ob

{
  "tracking_uri": "sqlite:////mnt/c/Users/SN23341J001185/Documents/MyWork/ICT/ICTYear4_1/MLop/public_teaching_mlaiops/course/MLOP-PJ/mlflow.db",
  "registry_scope": "local SQLite, not Azure ML",
  "run_id": "c269d7a94aa744ff8e6b76707bd8383a",
  "registered_name": "travel-style-segmentation",
  "registered_version": 1,
  "model_uri": "models:/travel-style-segmentation/1",
  "bundle_version": "abfd88e893ca",
  "bundle_sha256": "71f3ba41b4b964f5beb31e3d6e13b835b1dd2538aa663741d8837301be9dff13",
  "raw_sha256": "c07e7751f429ebe6225a77a32e61770dbb7197fd253f667e0434424adbaf7a33",
  "dvc_pointer_sha256": "c4daedad2c0789056894daca63b7e7ff5539d780663c507797db1c2c5b360c24",
  "dvc_status_clean": true,
  "registry_reload_passed": true,
  "training_repeated": false,
  "cloud_uploaded": false,
  "git_head": "03d9633d7be06133c72c3a94c0da33cec7e429cb",
  "git_dirty": true
}


## 2. Check the registry

Both checks below must pass. The registered model is downloaded from the local registry and its predictions must match the saved examples.

In [3]:
assert registry["dvc_status_clean"]
assert registry["registry_reload_passed"]
assert registry["bundle_version"] == json.loads((ROOT / "reports/bundle-manifest.json").read_text())["model_version"]
print("DVC data unchanged: passed")
print("Registered model reload: passed")
print("Registry scope:", registry["registry_scope"])


DVC data unchanged: passed
Registered model reload: passed
Registry scope: local SQLite, not Azure ML


## Output

`data/raw.dvc`, local MLflow data, `requirements-local.lock.txt`, and `reports/local-registry.json`. Continue to 06. `dvc.yaml` is prepared for later reproduction; this notebook does not run that training pipeline.